# 14 EMA Momentum Sweep

**Question:** How sensitive is SDD to the teacher EMA momentum?

DINO uses momentum ~0.996 but this was tuned for image classifiers.
In a diffusion model the teacher receives a different signal — we sweep
`[0.990, 0.996, 0.999]` and measure FID + linear probe accuracy.

**Expected finding:** Very high momentum (0.999) = stale teacher → poor
distillation signal. Very low (0.990) = unstable teacher → noisy gradients.
The sweet spot is the paper's main finding.

In [ ]:
from src.experiments import load_cfg, deep_update, run_ema_momentum_sweep
import matplotlib.pyplot as plt

base_cfg = load_cfg('configs/cifar10.yaml')
base_cfg = deep_update(base_cfg, {
    'wandb': {'enabled': True, 'tags': ['cifar10', 'ema_sweep']},
})

results = run_ema_momentum_sweep(base_cfg, momentum_values=[0.990, 0.996, 0.999])
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

x = results['teacher_momentum'].astype(str)

axes[0].bar(x, results['fid'], color='steelblue')
axes[0].set_xlabel('Teacher EMA momentum')
axes[0].set_ylabel('FID ↓')
axes[0].set_title('FID vs EMA momentum')
for i, v in enumerate(results['fid']):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=10)

axes[1].bar(x, results['linear_probe_acc'], color='darkorange')
axes[1].set_xlabel('Teacher EMA momentum')
axes[1].set_ylabel('Linear probe acc ↑')
axes[1].set_title('Representation quality vs EMA momentum')
for i, v in enumerate(results['linear_probe_acc']):
    axes[1].text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('outputs/figures/ema_momentum_sweep.png', dpi=150)
plt.show()

print('Best momentum (FID):       ', results.loc[results['fid'].idxmin(), 'teacher_momentum'])
print('Best momentum (probe acc): ', results.loc[results['linear_probe_acc'].idxmax(), 'teacher_momentum'])